# core

> Shared utilities and configuration for the segmentation library: reproducible seeding, device selection, and the `SegConfig` task descriptor.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import random
from dataclasses import dataclass

import numpy as np
import torch

## Reproducible seeding

Segmentation experiments must be reproducible: the same seed should give the same augmentations, the same weight init, and the same data shuffling. Let us first confirm that seeding the three RNGs we use (`random`, `numpy`, `torch`) makes two draws identical.

In [ ]:
# experiment: seed, draw, re-seed, draw again -> the two draws must match
random.seed(0); np.random.seed(0); torch.manual_seed(0)
draw_a = (random.random(), float(np.random.rand()), torch.rand(1).item())
random.seed(0); np.random.seed(0); torch.manual_seed(0)
draw_b = (random.random(), float(np.random.rand()), torch.rand(1).item())
print('draw_a:', draw_a)
print('draw_b:', draw_b)
print('identical:', draw_a == draw_b)

In [ ]:
#| export
def set_seed(
    seed: int = 42,        # value to seed every RNG with
    deterministic: bool = True,  # force cuDNN into deterministic (slower) mode
) -> None:
    """Seed Python, NumPy and PyTorch RNGs so a run is reproducible.

    Example:
        >>> set_seed(0)
        >>> a = torch.rand(1)
        >>> set_seed(0)
        >>> b = torch.rand(1)
        >>> bool((a == b).all())
        True
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
# test: two seeded draws are identical
set_seed(0); x = torch.rand(3)
set_seed(0); y = torch.rand(3)
assert torch.equal(x, y), 'set_seed did not make draws reproducible'

## Device selection

A tiny helper so every notebook picks the GPU when present and falls back to CPU otherwise.

In [ ]:
# experiment: what device will we train on?
print('cuda available:', torch.cuda.is_available())

In [ ]:
#| export
def get_device(
    prefer_cuda: bool = True,  # use CUDA when available
) -> torch.device:
    """Return the best available `torch.device`.

    Example:
        >>> isinstance(get_device(), torch.device)
        True
    """
    if prefer_cuda and torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

In [ ]:
# test
dev = get_device()
assert isinstance(dev, torch.device)
assert dev.type in ('cuda', 'cpu')

## `SegConfig`: the task descriptor

Everything downstream (model, losses, metrics, dataset) needs to know two numbers: how many input channels the images have, and how many classes the mask contains. We make binary segmentation the `num_classes == 1` special case rather than a separate code path, so the same modules serve binary pin-detection and multi-class segmentation alike.

In [ ]:
#| export
@dataclass
class SegConfig:
    """Describes a segmentation task end-to-end.

    `num_classes == 1` means binary segmentation with a single sigmoid logit.
    `num_classes > 1` means multi-class segmentation with softmax logits.

    Example:
        >>> cfg = SegConfig(in_channels=3, num_classes=1)
        >>> cfg.is_binary
        True
    """
    in_channels: int = 3      # number of channels in the model input image
    num_classes: int = 1      # 1 = binary (sigmoid); >1 = multi-class (softmax)
    ignore_index: int = -100  # mask label excluded from loss/metrics

    @property
    def is_binary(self) -> bool:
        "True when this is a single-foreground binary task."
        return self.num_classes == 1

In [ ]:
# test
binary = SegConfig(in_channels=3, num_classes=1)
assert binary.is_binary
multi = SegConfig(in_channels=4, num_classes=5)
assert not multi.is_binary
assert multi.in_channels == 4 and multi.num_classes == 5

## Platform diagnostics

Call `platform_info()` at the top of any notebook to catch cross-platform issues early (64-bit Linux vs Windows, CUDA presence, library versions).

> **Why this matters:** some OpenCV calls (e.g. `cv2.drawContours`) silently no-op on 64-bit Linux when passed `int64` arrays instead of `int32`. Printing the platform at notebook startup makes it obvious which OS you are on.

In [ ]:
# experiment: what platform are we on?
import platform, sys
print('OS         :', platform.system(), platform.release())
print('Python bits:', f'{sys.maxsize.bit_length() + 1}-bit')
import cv2, numpy as np
print('OpenCV     :', cv2.__version__)
print('NumPy      :', np.__version__)
print('intp dtype :', np.dtype(np.intp))   # int64 on 64-bit — causes silent cv2 bugs!

In [ ]:
#| export
def platform_info() -> dict:
    """Return OS, Python bitness and key CV library versions as a dict.

    Call once at notebook startup to confirm which platform you are on.
    Useful for catching silent cross-platform issues (e.g. OpenCV dtype behaviour
    differs between 64-bit Linux and Windows).

    Example:
        >>> info = platform_info()
        >>> info['os'] in ('Linux', 'Windows', 'Darwin')
        True
    """
    import platform as _pl
    import sys as _sys
    info: dict = {
        'os'         : _pl.system(),                       # 'Linux' | 'Windows' | 'Darwin'
        'os_release' : _pl.release(),
        'python'     : _sys.version.split()[0],
        'python_bits': f'{_sys.maxsize.bit_length() + 1}-bit',
    }
    try:
        import cv2 as _cv2;  info['opencv'] = _cv2.__version__
    except ImportError:
        pass
    try:
        import numpy as _np; info['numpy'] = _np.__version__
    except ImportError:
        pass
    try:
        import torch as _t
        info['torch']          = _t.__version__
        info['cuda_available'] = _t.cuda.is_available()
        if _t.cuda.is_available():
            info['cuda_device'] = _t.cuda.get_device_name(0)
    except ImportError:
        pass
    return info

In [ ]:
# test + demo
info = platform_info()
for k, v in info.items():
    print(f'{k:<20}: {v}')
assert 'os' in info and 'python_bits' in info

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()